<a href="https://colab.research.google.com/github/YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales/blob/main/Proyectos%20finales/Grupo%207/Grupo_7_Rosas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Grupo 7. Rosa Freedom: inteligencia comercial para priorizar mercados de exportación

**Asignatura:** Inteligencia en Negocios Globales — Universidad EAN
**Proyecto final — Grupo 7**
**Producto:** Rosas frescas cortadas, **HS 060311** (la variedad Freedom no se separa en las estadísticas; los indicadores se leen a nivel de categoría, como advierte la ficha)
**Ficha técnica de referencia:** *Equipo 7 — Ficha técnica del estudio* (septiembre de 2026)

---

## La pregunta del equipo

> ¿Qué mercados de destino ofrecen la mejor combinación entre ventaja comparativa revelada, participación y crecimiento de la demanda, especialización exportadora y riesgo controlable para la Rosa Freedom colombiana?

## Lo que calcula este cuaderno

Cada sección corresponde a un análisis que el equipo propuso en su ficha técnica. El cuaderno **calcula y grafica; no interpreta**. La interpretación es el trabajo del equipo.

| Métrica o análisis | Pregunta que responde |
|---|---|
| **Series por destino y CAGR** | ¿Hacia dónde van las rosas colombianas y cómo ha cambiado en diez años? |
| **HHI y CR3 de destinos** | ¿De cuántos mercados depende realmente la rosa colombiana? |
| **Balanza Comercial Relativa (BCR)** | ¿Colombia es exportador neto puro de rosas? |
| **Participación de mercado por destino** | ¿Qué porción de las importaciones de rosas de cada destino es colombiana? |
| **Valor unitario (USD/t)** | ¿A qué precio implícito vende Colombia frente a sus competidores y en cada destino? |
| **Cuota en la oferta mundial** | ¿Qué lugar ocupa Colombia entre los exportadores de rosas? |
| **RCA, RSCA y NRCA** | ¿Colombia exporta proporcionalmente más rosas que el resto del mundo? |
| **Índice de Lafay** | ¿Las rosas van mejor que el promedio comercial de Colombia? |
| **Matriz de decisión 0-100 con sensibilidad** | ¿Qué mercado priorizar, y qué tan robusta es esa respuesta? |

## Cómo usar este cuaderno

1. Ejecuta las celdas **en orden**, de arriba hacia abajo (`Entorno de ejecución → Ejecutar todas`). Viene en modo `"github"`: descarga sus propios datos del repositorio del curso, no tienes que subir nada.
2. Después de cada gráfica hay una celda que dice **Análisis del equipo**. Haz doble clic sobre ella y escribe la interpretación. Puedes agregar más celdas de texto donde quieras.
3. Las celdas marcadas **editable** contienen pesos o puntajes que el equipo debe ajustar con su propio criterio. Cámbialos y vuelve a ejecutar.
4. Al terminar: `Archivo → Descargar → Descargar .ipynb` y sube el archivo al aula virtual.

Todas las tablas y figuras se guardan además en la carpeta `salidas/` (panel izquierdo de Colab) para que las uses en el informe.

---
# 0. Preparación del entorno

In [ ]:
import os
import io
import csv
import glob
import time
import shutil
import zipfile
import urllib.request
import urllib.error
import urllib.parse

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 170)

# Paleta de colores del curso (validada para lectura accesible en pantalla e impresion)
AZUL      = "#2a78d6"    # serie principal / pais del caso
ROJO      = "#e34948"    # valores negativos (par divergente con el azul)
NARANJA   = "#eb6834"    # elemento destacado
VERDE     = "#3f8f6b"    # segunda serie categorica
GRIS_MID  = "#c3c2b7"    # resto / punto neutro
TINTA     = "#0b0b0b"
GRIS_TEXT = "#52514e"
GRIS_EJE  = "#898781"
REJILLA   = "#e1e0d9"

# Orden fijo de colores para series categoricas: nunca se reciclan, lo que sobra va a "Otros" en gris
CATEGORIAS = [AZUL, NARANJA, VERDE, ROJO]

CARPETA_SALIDA = "salidas"
os.makedirs(CARPETA_SALIDA, exist_ok=True)


def estilo(ax, titulo, subtitulo="", fuente="", eje_y="", eje_x="", rejilla="y"):
    """Aplica el estilo de graficas del curso: titulo a la izquierda, sin marco, rejilla suave, fuente al pie."""
    ax.set_title(titulo + ("\n" + subtitulo if subtitulo else ""),
                 fontsize=13, color=TINTA, loc="left", pad=14)
    ax.set_ylabel(eje_y, fontsize=10, color=GRIS_TEXT)
    ax.set_xlabel(eje_x, fontsize=10, color=GRIS_TEXT)
    if rejilla == "y":
        ax.yaxis.grid(True, color=REJILLA, linewidth=0.8)
    elif rejilla == "x":
        ax.xaxis.grid(True, color=REJILLA, linewidth=0.8)
    ax.set_axisbelow(True)
    for lado in ["top", "right", "left"]:
        ax.spines[lado].set_visible(False)
    ax.spines["bottom"].set_color(GRIS_EJE)
    ax.tick_params(colors=GRIS_EJE, labelsize=9)
    if fuente:
        ax.figure.text(0.01, -0.03, "Fuente: " + fuente, fontsize=8, color=GRIS_EJE, ha="left")


def guardar(fig, nombre):
    """Muestra la figura y la guarda como PNG en la carpeta de salidas."""
    ruta = os.path.join(CARPETA_SALIDA, nombre + ".png")
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.show()
    print("Figura guardada en", ruta)


def exportar(tabla, nombre):
    """Guarda una tabla como CSV en la carpeta de salidas y la devuelve para mostrarla."""
    tabla.to_csv(os.path.join(CARPETA_SALIDA, nombre + ".csv"), index=False, encoding="utf-8-sig")
    return tabla


def fmt_miles(x, _=None):
    """Formato de eje para valores en miles de USD: 1200 -> 1.2 mn ; 850 -> 850 k."""
    if abs(x) >= 1_000_000:
        return f"{x / 1_000_000:.1f} mil mn"
    if abs(x) >= 1_000:
        return f"{x / 1_000:.1f} mn"
    return f"{x:.0f} k"


def fmt_unidades(x, _=None):
    """Formato de eje para valores en unidades: 1.5e9 -> 1.5 mil mn ; 2.4e6 -> 2.4 mn ; 850000 -> 850 k."""
    if abs(x) >= 1e9:
        return f"{x / 1e9:.1f} mil mn"
    if abs(x) >= 1e6:
        return f"{x / 1e6:.1f} mn"
    if abs(x) >= 1e3:
        return f"{x / 1e3:.0f} k"
    return f"{x:.0f}"


def leyenda_fuera(ax):
    """Leyenda a la derecha del grafico, para que no tape barras ni la nota de fuente."""
    ax.legend(frameon=False, fontsize=8.5, loc="upper left", bbox_to_anchor=(1.01, 1))


print("pandas:", pd.__version__, "| numpy:", np.__version__)

In [ ]:
# ============================================================
#  CONFIGURACION: cambia solo esta celda si lo necesitas
# ============================================================

MODO = "github"         # opciones: "github" | "subir" | "local"

REPO_CURSO = "YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales"
RAMA = "main"

# Archivos que necesita este cuaderno, con su ruta dentro del repositorio del curso.
# Los del grupo estan en "Proyectos finales/Grupo 7/datos/"; los demas son datos del curso en "data/".
ARCHIVOS_REPO = {
    "exp_serie": "Proyectos finales/Grupo 7/datos/Datos_proyectobrief/colombias-exports-to-world-by-importer_060311.csv",
    "exp_2025": "Proyectos finales/Grupo 7/datos/Datos_proyectobrief/colombias-exports-to-world-in-2025-by-importer_060311.csv",
    "exp_all": "Proyectos finales/Grupo 7/datos/Datos_proyectobrief/colombias-exports-to-world-in-2025-by-importer_all.csv",
    "imp_2025": "Proyectos finales/Grupo 7/datos/Datos_proyectobrief/colombias-imports-from-world-in-2025-by-exporter_060311.csv",
    "imp_all": "Proyectos finales/Grupo 7/datos/Datos_proyectobrief/colombias-imports-from-world-in-2025-by-exporter_all.csv",
    "mundo_exp": "Proyectos finales/Grupo 7/datos/Datos_proyectobrief/exporting-countries-in-2025_060311.csv",
    "canasta_co_exp": "data/co_exp_productos_hs2_serie.xls",
    "canasta_co_imp": "data/co_imp_productos_hs2_serie.csv",
    "canasta_mundo_exp": "data/mundo_exp_productos_hs2_serie.csv",
}

RUTA_LOCAL = "../.."    # solo si MODO = "local": raiz del repositorio, corriendo desde la carpeta del grupo

print(f"Modo seleccionado: {MODO}  |  {len(ARCHIVOS_REPO)} archivos")

In [ ]:
# ============================================================
#  Ejecuta esta celda tal cual: consigue los datos segun el modo
# ============================================================

def pedir(url, intentos=5):
    """Descarga una URL, reintentando si el servidor pide esperar (HTTP 429 de GitHub en Colab)."""
    for intento in range(intentos):
        try:
            peticion = urllib.request.Request(url, headers={"User-Agent": "cuaderno-ean"})
            with urllib.request.urlopen(peticion, timeout=90) as respuesta:
                return respuesta.read()
        except urllib.error.HTTPError as error:
            if error.code not in (403, 429, 500, 502, 503) or intento == intentos - 1:
                raise
            espera = int(error.headers.get("Retry-After") or 0) or 2 ** intento
            print(f"    servidor ocupado (HTTP {error.code}); reintento en {espera} s")
            time.sleep(espera)
        except urllib.error.URLError:
            if intento == intentos - 1:
                raise
            time.sleep(2 ** intento)


def descargar_datos(rutas_repo, destino):
    """Trae los archivos del repositorio a la carpeta destino, en una sola peticion (zip del repo).

    Cada archivo se guarda por su nombre, sin carpetas. Si ya existe no se vuelve a bajar.
    Si el zip falla, baja los archivos uno por uno desde raw.githubusercontent.com.
    """
    os.makedirs(destino, exist_ok=True)
    faltan = [r for r in rutas_repo if not os.path.exists(os.path.join(destino, os.path.basename(r)))]
    if not faltan:
        print(f"Los {len(rutas_repo)} archivos ya estaban descargados.")
        return
    try:
        print(f"Descargando {len(faltan)} archivos en una sola peticion...\n")
        comprimido = pedir(f"https://codeload.github.com/{REPO_CURSO}/zip/refs/heads/{RAMA}")
        with zipfile.ZipFile(io.BytesIO(comprimido)) as paquete:
            for miembro in paquete.namelist():
                relativo = miembro.split("/", 1)[1] if "/" in miembro else miembro
                if relativo in faltan:
                    with paquete.open(miembro) as origen, \
                         open(os.path.join(destino, os.path.basename(relativo)), "wb") as salida:
                        shutil.copyfileobj(origen, salida)
                    print(f"  extraido: {os.path.basename(relativo)}")
    except Exception as error:
        print(f"\n  El paquete fallo ({type(error).__name__}). Voy archivo por archivo.\n")
        for relativo in faltan:
            url = f"https://raw.githubusercontent.com/{REPO_CURSO}/{RAMA}/" + urllib.parse.quote(relativo)
            with open(os.path.join(destino, os.path.basename(relativo)), "wb") as salida:
                salida.write(pedir(url))
            print(f"  descargado: {os.path.basename(relativo)}")
            time.sleep(0.5)
    perdidos = [r for r in rutas_repo if not os.path.exists(os.path.join(destino, os.path.basename(r)))]
    if perdidos:
        raise FileNotFoundError(f"No se pudieron descargar: {perdidos}. Espera un minuto y vuelve a ejecutar.")


if MODO == "github":
    RUTA_BASE = "datos_crudos"
    descargar_datos(list(ARCHIVOS_REPO.values()), RUTA_BASE)

    def ruta(clave):
        return os.path.join(RUTA_BASE, os.path.basename(ARCHIVOS_REPO[clave]))

elif MODO == "subir":
    from google.colab import files
    print("Sube estos archivos:\n  " + "\n  ".join(os.path.basename(v) for v in ARCHIVOS_REPO.values()))
    files.upload()
    RUTA_BASE = "/content"

    def ruta(clave):
        encontrados = glob.glob(os.path.join(RUTA_BASE, "**", os.path.basename(ARCHIVOS_REPO[clave])), recursive=True)
        if not encontrados:
            raise FileNotFoundError(f"Falta el archivo {os.path.basename(ARCHIVOS_REPO[clave])}")
        return encontrados[0]

else:  # local
    RUTA_BASE = RUTA_LOCAL

    def ruta(clave):
        return os.path.join(RUTA_BASE, ARCHIVOS_REPO[clave])

for clave in ARCHIVOS_REPO:
    estado = "ok" if os.path.exists(ruta(clave)) else "FALTA"
    print(f"  {estado:5s} {clave:14s} -> {os.path.basename(ARCHIVOS_REPO[clave])}")

In [ ]:
# ============================================================
#  Lectores: cada funcion resuelve las trampas de un tipo de archivo
# ============================================================

def a_numero(serie):
    """Convierte a numero una columna que viene como texto (miles con coma, simbolos, espacios)."""
    return pd.to_numeric(
        pd.Series(serie).astype(str)
                        .str.replace(",", "", regex=False)
                        .str.replace(r"[^0-9.\-]", "", regex=True)
                        .replace("", np.nan),
        errors="coerce",
    )


# Nombres cortos en espanol para las columnas de Trade Map
COLUMNAS_TM = {
    "Value (kUSD)": "valor_kusd",
    "Balance (kUSD)": "balanza_kusd",
    "Quantity": "cantidad",
    "Quantity Unit": "unidad_cantidad",
    "Unit Value": "valor_unitario",
    "Unit Value Unit": "unidad_valor_unitario",
    "Share (%)": "participacion_pct",
    "Share Partner Country (%)": "cuota_en_socio_pct",
    "Share World (%)": "participacion_mundial_pct",
    "Ranking Partners": "ranking_socio",
    "Growth Value 5Y (%)": "crec_valor_5a_pct",
    "Growth Value 2Y (%)": "crec_valor_2a_pct",
    "Growth Value Partners 5Y (%)": "crec_importaciones_socio_5a_pct",
    "Growth Quantity 5Y (%)": "crec_cantidad_5a_pct",
}

AGREGADOS_NO_PAIS = ["Zona franca", "Zonas francas", "Áreas Nes", "Areas, nes", "Zona Nep", "Free Zones"]


def leer_trademap(ruta_archivo):
    """Lee una tabla de indicadores de Trade Map (corte de un anio, formato largo).

    Trampas que resuelve: una columna sin nombre en el encabezado, codigos de pais con cero
    inicial que pandas convertiria a entero, valor unitario con 28 decimales, y la fila
    "Mundo" mezclada con los paises. La columna `pais` queda lista para usar.
    """
    tabla = pd.read_csv(ruta_archivo, dtype={"reporterCd": str, "partnerCd": str, "productCd": str},
                        encoding="utf-8-sig")
    return limpiar_trademap(tabla)


def limpiar_trademap(tabla):
    """Limpieza comun a toda tabla de indicadores de Trade Map, venga de CSV o de Excel."""
    tabla = tabla.loc[:, ~tabla.columns.str.startswith("Unnamed")]
    for columna in ("reporterCd", "partnerCd"):
        tabla[columna] = tabla[columna].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(3)
    tabla = tabla.rename(columns=COLUMNAS_TM)
    for columna in COLUMNAS_TM.values():
        if columna in tabla.columns and columna not in ("unidad_cantidad", "unidad_valor_unitario"):
            tabla[columna] = a_numero(tabla[columna])
    if "valor_unitario" in tabla.columns:
        tabla["valor_unitario"] = tabla["valor_unitario"].round(2)
    # Si todos los socios son "Mundo", la tabla es una lista de paises (reporter); si no, es una lista de socios
    if (tabla["partnerCd"] == "000").all():
        tabla["codigo"], tabla["pais"] = tabla["reporterCd"], tabla["reporterLabel"]
    else:
        tabla["codigo"], tabla["pais"] = tabla["partnerCd"], tabla["partnerLabel"]
    return tabla


def separar_mundo(tabla):
    """Devuelve (fila Mundo, tabla solo con paises). Excluye agregados que no son paises."""
    es_mundo = tabla["codigo"] == "000"
    mundo = tabla[es_mundo].iloc[0]
    paises = tabla[~es_mundo & ~tabla["pais"].isin(AGREGADOS_NO_PAIS)].reset_index(drop=True)
    return mundo, paises


def leer_serie_trademap(ruta_archivo):
    """Lee la serie anual por socio (un anio por columna) y la devuelve en formato largo.

    Columnas de salida: codigo, pais, anio, valor_kusd. Incluye la fila Mundo (codigo 000).
    """
    tabla = pd.read_csv(ruta_archivo, dtype=str, encoding="utf-8-sig")
    columnas_anio = [c for c in tabla.columns if c[:4].isdigit()]
    tabla = tabla.rename(columns={c: c[:4] for c in columnas_anio})
    anios = [c[:4] for c in columnas_anio]
    for anio in anios:
        tabla[anio] = a_numero(tabla[anio])
    if (tabla["partnerCd"] == "000").all():
        tabla["codigo"], tabla["pais"] = tabla["reporterCd"], tabla["reporterLabel"]
    else:
        tabla["codigo"], tabla["pais"] = tabla["partnerCd"], tabla["partnerLabel"]
    largo = tabla.melt(id_vars=["codigo", "pais"], value_vars=anios, var_name="anio", value_name="valor_kusd")
    largo["anio"] = largo["anio"].astype(int)
    return largo.sort_values(["anio", "valor_kusd"], ascending=[True, False]).reset_index(drop=True)


def leer_banco_mundial(ruta_archivo, nombre_indicador):
    """Lee un archivo del Banco Mundial: cuatro filas de metadatos antes del encabezado y un anio por columna."""
    tabla = pd.read_csv(ruta_archivo, skiprows=4, encoding="utf-8-sig")
    tabla = tabla.loc[:, ~tabla.columns.str.startswith("Unnamed")]
    anios = [c for c in tabla.columns if c.isdigit()]
    largo = tabla.melt(id_vars=["Country Name", "Country Code"], value_vars=anios,
                       var_name="anio", value_name=nombre_indicador)
    largo["anio"] = largo["anio"].astype(int)
    return largo.rename(columns={"Country Name": "pais", "Country Code": "iso3"})


ANIOS_CANASTA = [str(a) for a in range(2021, 2026)]


def leer_canasta(ruta_archivo):
    """Lee la canasta por capitulo HS del curso (97 capitulos + TOTAL, 2021-2025).

    Funciona igual si el archivo es un .csv o un .xls que en realidad es HTML.
    Trampas: apostrofe delante del codigo y miles separados por coma como texto.
    """
    with open(ruta_archivo, encoding="utf-8", errors="ignore") as f:
        primeras_letras = f.read(200).lstrip().lower()
    if primeras_letras.startswith("<"):
        tabla = max(pd.read_html(ruta_archivo), key=lambda t: t.shape[0])
    else:
        tabla = pd.read_csv(ruta_archivo)
    tabla = tabla.iloc[:, -7:]
    tabla.columns = ["codigo", "producto"] + ANIOS_CANASTA
    tabla["codigo"] = tabla["codigo"].astype(str).str.strip().str.lstrip("'").str.strip()
    for anio in ANIOS_CANASTA:
        tabla[anio] = a_numero(tabla[anio])
    return tabla.dropna(subset=["2025"]).reset_index(drop=True)


def fila_canasta(canasta, codigo):
    """Devuelve la serie 2021-2025 (en USD miles) de un codigo de la canasta: un capitulo HS2 o "TOTAL"."""
    fila = canasta[canasta["codigo"] == codigo]
    if fila.empty:
        raise KeyError(f"No encontre el codigo {codigo} en la canasta")
    return fila.iloc[0][ANIOS_CANASTA].astype(float)


print("Lectores listos.")

In [ ]:
# ============================================================
#  Formulas: las mismas de los cuadernos 2, 3 y 4 del curso
# ============================================================

def hhi(valores):
    """Indice de Herfindahl-Hirschman: suma de participaciones al cuadrado. Entre 0 y 1."""
    valores = np.asarray(valores, dtype=float)
    valores = valores[~np.isnan(valores)]
    valores = valores[valores > 0]
    if valores.sum() == 0:
        return np.nan
    participaciones = valores / valores.sum()
    return float(np.sum(participaciones ** 2))


def numeros_equivalentes(indice_hhi):
    """Cuantos destinos del mismo tamano equivaldrian a esta reparticion: 1 / HHI."""
    return 1 / indice_hhi

def cr_n(valores, n):
    """Razon de concentracion CR_n: participacion conjunta (%) de los n mayores."""
    valores = np.sort(np.asarray(valores, dtype=float))[::-1]
    valores = valores[~np.isnan(valores)]
    return float(valores[:n].sum() / valores.sum() * 100)

def ibcr(exportaciones, importaciones):
    """Indice de Balanza Comercial Relativa: (X - M) / (X + M), entre -1 y +1."""
    exportaciones = np.asarray(exportaciones, dtype=float)
    importaciones = np.asarray(importaciones, dtype=float)
    comercio_total = exportaciones + importaciones
    return np.where(comercio_total > 0, (exportaciones - importaciones) / comercio_total, np.nan)

def rca_balassa(x_pais, x_pais_total, x_mundo, x_mundo_total):
    """Ventaja Comparativa Revelada de Balassa (1965). Mayor que 1 = especializacion revelada."""
    participacion_pais  = np.asarray(x_pais, dtype=float)  / np.asarray(x_pais_total, dtype=float)
    participacion_mundo = np.asarray(x_mundo, dtype=float) / np.asarray(x_mundo_total, dtype=float)
    return participacion_pais / participacion_mundo


def rsca_laursen(rca):
    """RCA simetrico de Laursen: (RCA - 1) / (RCA + 1), entre -1 y +1, con 0 como punto neutro."""
    rca = np.asarray(rca, dtype=float)
    return (rca - 1) / (rca + 1)

def nrca(x_pais, x_pais_total, x_mundo, x_mundo_total):
    """NRCA de Yu, Cai y Leung (2009): distancia entre la exportacion observada y la neutral. Suma cero."""
    x_pais, x_mundo = np.asarray(x_pais, dtype=float), np.asarray(x_mundo, dtype=float)
    x_pais_total, x_mundo_total = float(x_pais_total), float(x_mundo_total)
    observado = x_pais / x_mundo_total
    neutral   = (x_pais_total * x_mundo) / (x_mundo_total ** 2)
    return observado - neutral

def lafay(x_pais, m_pais):
    """Indice de Lafay (1992) sobre una canasta completa: positivo = el producto va mejor que el promedio del pais."""
    x_pais = np.asarray(x_pais, dtype=float)
    m_pais = np.asarray(m_pais, dtype=float)
    comercio_producto = x_pais + m_pais
    comercio_total    = comercio_producto.sum()
    ibcr_producto = np.where(comercio_producto > 0, (x_pais - m_pais) / comercio_producto, 0)
    ibcr_nacional = (x_pais - m_pais).sum() / comercio_total
    peso_producto = comercio_producto / comercio_total
    return 100 * (ibcr_producto - ibcr_nacional) * peso_producto

def cagr(valor_inicial, valor_final, anios):
    """Tasa de crecimiento anual compuesto (%) entre dos valores separados por `anios` anios."""
    valor_inicial, valor_final = float(valor_inicial), float(valor_final)
    if valor_inicial <= 0 or valor_final <= 0 or anios <= 0:
        return np.nan
    return ((valor_final / valor_inicial) ** (1 / anios) - 1) * 100

def matriz_multicriterio(puntajes, pesos):
    """Pondera una tabla de puntajes (filas = criterios, columnas = mercados) con un diccionario de pesos.

    Devuelve la contribucion de cada criterio por mercado y el total, ordenado de mayor a menor.
    Los pesos se normalizan para que sumen 1, asi que puedes escribirlos en % o en fracciones.
    """
    pesos = pd.Series(pesos, dtype=float)
    pesos = pesos / pesos.sum()
    contribucion = puntajes.mul(pesos, axis=0)
    resultado = contribucion.T
    resultado["Total"] = resultado.sum(axis=1)
    return resultado.sort_values("Total", ascending=False)

print('Formulas listas.')

---
# 1. Los datos

Seis descargas de Trade Map del equipo (HS 060311 y el total de todos los productos) y tres archivos del curso con la canasta por capítulo HS2, que aportan los totales mundiales que faltaban para Balassa y Lafay.

In [ ]:
serie = leer_serie_trademap(ruta("exp_serie"))              # Colombia exporta rosas, por destino, 2016-2025
exp_2025 = leer_trademap(ruta("exp_2025"))                  # lo mismo, corte 2025 con indicadores
exp_all  = leer_trademap(ruta("exp_all"))                   # Colombia exporta TODO, por destino, 2025
imp_2025 = leer_trademap(ruta("imp_2025"))                  # Colombia importa rosas, por origen, 2025
imp_all  = leer_trademap(ruta("imp_all"))                   # Colombia importa TODO, 2025
mundo_exp = leer_trademap(ruta("mundo_exp"))                # exportadores mundiales de rosas, 2025

mundo_2025, destinos_2025 = separar_mundo(exp_2025)
mundo_oferta, exportadores = separar_mundo(mundo_exp)

print(f"Colombia exporto rosas por USD {mundo_2025['valor_kusd']:,.0f} miles en 2025 a {len(destinos_2025)} destinos")
print(f"El mundo exporto rosas por USD {mundo_oferta['valor_kusd']:,.0f} miles; {len(exportadores)} paises exportadores")
destinos_2025[["pais", "valor_kusd", "cantidad", "valor_unitario", "participacion_pct", "cuota_en_socio_pct", "crec_valor_5a_pct"]].head(10)

In [ ]:
canasta_co_exp    = leer_canasta(ruta("canasta_co_exp"))
canasta_co_imp    = leer_canasta(ruta("canasta_co_imp"))
canasta_mundo_exp = leer_canasta(ruta("canasta_mundo_exp"))

print("Capitulo 06 (plantas vivas y flores) en la canasta exportadora colombiana, USD miles:")
print(fila_canasta(canasta_co_exp, "06").round(0).to_string())
print("\nExportaciones mundiales totales (TOTAL), USD miles:")
print(fila_canasta(canasta_mundo_exp, "TOTAL").round(0).to_string())

---
# 2. Series por destino, 2016-2025, y CAGR

In [ ]:
paises_serie = serie[serie["codigo"] != "000"]
top4 = paises_serie[paises_serie["anio"] == 2025].nlargest(4, "valor_kusd")["pais"].tolist()

series_top = (paises_serie.assign(grupo=lambda d: np.where(d["pais"].isin(top4), d["pais"], "Otros"))
                          .groupby(["anio", "grupo"])["valor_kusd"].sum()
                          .unstack("grupo")[top4 + ["Otros"]])
exportar(series_top.reset_index(), "g7_series_destinos")
series_top

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colores = CATEGORIAS + [GRIS_MID]
for color, columna in zip(colores, series_top.columns):
    ax.plot(series_top.index, series_top[columna], color=color, linewidth=2.2, marker="o", markersize=5, label=columna)
ax.yaxis.set_major_formatter(plt.FuncFormatter(fmt_miles))
ax.set_xticks(series_top.index)
ax.legend(frameon=False, fontsize=9, loc="upper left")
estilo(ax, "Exportaciones colombianas de rosas por destino, 2016-2025",
       "HS 060311, USD (los valores del archivo vienen en miles)", "Trade Map (ITC), 2025.", eje_y="USD")
guardar(fig, "g7_series_destinos")

In [ ]:
# CAGR 2016-2025 y 2020-2025 por destino (los diez mayores de 2025)
ancho = paises_serie.pivot(index="pais", columns="anio", values="valor_kusd")
top10 = ancho[2025].nlargest(10).index
tabla_cagr = pd.DataFrame({
    "valor_2016": ancho.loc[top10, 2016],
    "valor_2025": ancho.loc[top10, 2025],
    "cagr_2016_2025_pct": [cagr(ancho.loc[p, 2016], ancho.loc[p, 2025], 9) for p in top10],
    "cagr_2020_2025_pct": [cagr(ancho.loc[p, 2020], ancho.loc[p, 2025], 5) for p in top10],
}).reset_index()
exportar(tabla_cagr, "g7_cagr_destinos")
tabla_cagr

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
orden = tabla_cagr.sort_values("cagr_2016_2025_pct")
colores = [ROJO if v < 0 else AZUL for v in orden["cagr_2016_2025_pct"]]
ax.barh(orden["pais"], orden["cagr_2016_2025_pct"], color=colores, height=0.6)
for y, v in enumerate(orden["cagr_2016_2025_pct"]):
    ax.annotate(f"{v:+.1f} %", (v, y), textcoords="offset points", xytext=(4 if v >= 0 else -4, 0),
                ha="left" if v >= 0 else "right", va="center", fontsize=9, color=GRIS_TEXT)
ax.axvline(0, color=GRIS_EJE, linewidth=0.8)
estilo(ax, "Crecimiento anual compuesto por destino, 2016-2025",
       "Diez mayores destinos de 2025", "Trade Map (ITC), 2025.", eje_x="CAGR (%)", rejilla="x")
guardar(fig, "g7_cagr_destinos")

**Análisis del equipo:** ¿Qué destinos crecen, cuáles se estancan, y qué cambió después de 2020?

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 3. Concentración de destinos: HHI, CR3 y número equivalente

In [ ]:
concentracion = (paises_serie.groupby("anio")["valor_kusd"]
                 .agg(HHI=hhi, CR3=lambda v: cr_n(v, 3), destinos_activos=lambda v: int((v > 0).sum()))
                 .reset_index())
concentracion["numero_equivalente"] = numeros_equivalentes(concentracion["HHI"])
exportar(concentracion, "g7_hhi_destinos")
concentracion

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(concentracion["anio"], concentracion["HHI"], color=AZUL, linewidth=2.4, marker="o", markersize=7)
for _, fila in concentracion.iterrows():
    ax.annotate(f"{fila['HHI']:.3f}", (fila["anio"], fila["HHI"]), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=9, color=GRIS_TEXT)
ax.axhspan(0.18, 1, color=ROJO, alpha=0.06)
ax.axhspan(0.15, 0.18, color=NARANJA, alpha=0.06)
ax.set_ylim(0, max(0.8, concentracion["HHI"].max() * 1.15))
ax.set_xticks(concentracion["anio"])
estilo(ax, "HHI de destinos de las rosas colombianas, 2016-2025",
       "Bandas: > 0,18 concentrado; 0,15-0,18 moderado; < 0,15 fragmentado", "Trade Map (ITC), 2025.", eje_y="HHI")
guardar(fig, "g7_hhi_destinos")

In [ ]:
# Participacion por destino en 2025
participacion = destinos_2025.nlargest(12, "valor_kusd")[["pais", "valor_kusd", "participacion_pct"]]
fig, ax = plt.subplots(figsize=(9, 5.5))
orden = participacion.sort_values("participacion_pct")
ax.barh(orden["pais"], orden["participacion_pct"], color=AZUL, height=0.6)
for y, v in enumerate(orden["participacion_pct"]):
    ax.annotate(f"{v:.1f} %", (v, y), textcoords="offset points", xytext=(4, 0), va="center", fontsize=9, color=GRIS_TEXT)
estilo(ax, "Participación de cada destino en las exportaciones colombianas de rosas, 2025",
       f"HHI = {concentracion.iloc[-1]['HHI']:.3f}  |  CR3 = {concentracion.iloc[-1]['CR3']:.1f} %  |  número equivalente = {concentracion.iloc[-1]['numero_equivalente']:.1f}",
       "Trade Map (ITC), 2025.", eje_x="% del valor exportado", rejilla="x")
guardar(fig, "g7_participacion_destinos_2025")

**Análisis del equipo:** ¿La concentración es un riesgo o una fortaleza para la Rosa Freedom? ¿Cambió la tendencia?

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 4. Balanza Comercial Relativa

In [ ]:
mundo_imp_2025, _ = separar_mundo(imp_2025)
bcr_rosas = float(ibcr(mundo_2025["valor_kusd"], mundo_imp_2025["valor_kusd"]))

# La misma medida para el capitulo 06 completo (flores y plantas), 2021-2025, con la canasta del curso
x06, m06 = fila_canasta(canasta_co_exp, "06"), fila_canasta(canasta_co_imp, "06")
bcr_serie = pd.DataFrame({"anio": [int(a) for a in ANIOS_CANASTA], "X_cap06": x06.values, "M_cap06": m06.values})
bcr_serie["BCR_cap06"] = ibcr(bcr_serie["X_cap06"], bcr_serie["M_cap06"])

print(f"BCR rosas (HS 060311), 2025: X = {mundo_2025['valor_kusd']:,.0f}  M = {mundo_imp_2025['valor_kusd']:,.0f}  ->  BCR = {bcr_rosas:+.4f}")
exportar(bcr_serie, "g7_bcr")
bcr_serie

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(bcr_serie["anio"], bcr_serie["BCR_cap06"], color=VERDE, linewidth=2.2, marker="o", markersize=6, label="Capítulo 06 (flores y plantas)")
ax.scatter([2025], [bcr_rosas], color=AZUL, s=70, zorder=3, label="Rosas HS 060311, 2025")
ax.annotate(f"{bcr_rosas:+.3f}", (2025, bcr_rosas), textcoords="offset points", xytext=(8, -4), fontsize=9, color=GRIS_TEXT)
ax.set_ylim(-1.05, 1.05)
ax.axhline(0, color=GRIS_EJE, linewidth=0.8)
ax.set_xticks(bcr_serie["anio"])
ax.legend(frameon=False, fontsize=9, loc="lower left")
estilo(ax, "Balanza Comercial Relativa: rosas y capítulo 06", "+1 = exportador neto puro; -1 = importador neto puro",
       "Trade Map (ITC), 2025.", eje_y="BCR")
guardar(fig, "g7_bcr")

**Análisis del equipo:**

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 5. Participación de mercado en cada destino

La columna `cuota_en_socio_pct` de Trade Map es exactamente la definición de la ficha: exportaciones de Colombia al destino sobre las importaciones totales de rosas de ese destino.

In [ ]:
mercados = destinos_2025.nlargest(15, "valor_kusd")[
    ["pais", "valor_kusd", "cuota_en_socio_pct", "ranking_socio", "crec_valor_5a_pct", "crec_importaciones_socio_5a_pct", "valor_unitario"]]
exportar(mercados, "g7_participacion_mercado")
mercados

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
orden = mercados.dropna(subset=["cuota_en_socio_pct"]).sort_values("cuota_en_socio_pct")
ax.barh(orden["pais"], orden["cuota_en_socio_pct"], color=AZUL, height=0.6)
for y, (v, r) in enumerate(zip(orden["cuota_en_socio_pct"], orden["ranking_socio"])):
    ax.annotate(f"{v:.1f} %  (proveedor n.º {r:.0f})", (v, y), textcoords="offset points", xytext=(4, 0), va="center", fontsize=9, color=GRIS_TEXT)
estilo(ax, "Cuota de Colombia en las importaciones de rosas de cada destino, 2025",
       "Quince mayores destinos por valor exportado", "Trade Map (ITC), 2025.", eje_x="% de las importaciones del destino", rejilla="x")
guardar(fig, "g7_cuota_en_destino")

In [ ]:
# Cuota actual frente a crecimiento de la demanda del destino (importaciones totales de rosas del socio, 5 anios)
fig, ax = plt.subplots(figsize=(9, 6))
puntos = mercados.dropna(subset=["cuota_en_socio_pct", "crec_importaciones_socio_5a_pct"])
tamanos = puntos["valor_kusd"] / puntos["valor_kusd"].max() * 900 + 40
ax.scatter(puntos["cuota_en_socio_pct"], puntos["crec_importaciones_socio_5a_pct"], s=tamanos, color=AZUL, alpha=0.55, edgecolor="white", linewidth=1.5)
for _, p in puntos.iterrows():
    ax.annotate(p["pais"], (p["cuota_en_socio_pct"], p["crec_importaciones_socio_5a_pct"]),
                textcoords="offset points", xytext=(6, 4), fontsize=8.5, color=GRIS_TEXT)
ax.axhline(puntos["crec_importaciones_socio_5a_pct"].median(), color=GRIS_EJE, linewidth=0.8, linestyle="--")
ax.axvline(puntos["cuota_en_socio_pct"].median(), color=GRIS_EJE, linewidth=0.8, linestyle="--")
estilo(ax, "Cuota de Colombia frente al crecimiento de las importaciones de cada destino",
       "Tamaño del punto = valor exportado en 2025. Líneas punteadas = medianas", "Trade Map (ITC), 2025.",
       eje_x="Cuota de Colombia en el destino (%)", eje_y="Crecimiento de las importaciones del destino, 5 años (%)")
guardar(fig, "g7_cuota_vs_crecimiento")

**Análisis del equipo:** ¿En qué cuadrante cae cada mercado candidato (Estados Unidos, Canadá, Reino Unido, Países Bajos, Japón)?

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 6. Valor unitario

Como advierte la ficha, el valor unitario **no equivale a margen ni a precio neto**: es valor FOB dividido por toneladas.

In [ ]:
# Colombia frente a los grandes exportadores (se excluyen los que no reportan cantidad)
competidores = exportadores.nlargest(10, "valor_kusd")
competidores = competidores[(competidores["cantidad"] > 0) & competidores["valor_unitario"].notna()]
competidores = competidores[["pais", "valor_kusd", "cantidad", "valor_unitario", "participacion_mundial_pct"]]
exportar(competidores, "g7_valor_unitario_competidores")
competidores

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
orden = competidores.sort_values("valor_unitario")
colores = [NARANJA if p == "Colombia" else GRIS_MID for p in orden["pais"]]
ax.barh(orden["pais"], orden["valor_unitario"], color=colores, height=0.6)
for y, v in enumerate(orden["valor_unitario"]):
    ax.annotate(f"{v:,.0f}", (v, y), textcoords="offset points", xytext=(4, 0), va="center", fontsize=9, color=GRIS_TEXT)
estilo(ax, "Valor unitario de las exportaciones de rosas: Colombia frente a los grandes exportadores, 2025",
       "USD por tonelada, valor FOB / cantidad", "Trade Map (ITC), 2025.", eje_x="USD/t", rejilla="x")
guardar(fig, "g7_valor_unitario_competidores")

In [ ]:
# Valor unitario por destino de Colombia
vu_destinos = destinos_2025.nlargest(12, "valor_kusd")
vu_destinos = vu_destinos[vu_destinos["cantidad"] > 0][["pais", "valor_kusd", "cantidad", "valor_unitario"]]
fig, ax = plt.subplots(figsize=(9, 5.5))
orden = vu_destinos.sort_values("valor_unitario")
ax.barh(orden["pais"], orden["valor_unitario"], color=AZUL, height=0.6)
ax.axvline(mundo_2025["valor_unitario"], color=NARANJA, linewidth=1.5, linestyle="--")
ax.annotate(f"promedio Colombia {mundo_2025['valor_unitario']:,.0f}", (mundo_2025["valor_unitario"], len(orden) - 0.4),
            fontsize=9, color=NARANJA, ha="left", xytext=(4, 0), textcoords="offset points")
for y, v in enumerate(orden["valor_unitario"]):
    ax.annotate(f"{v:,.0f}", (v, y), textcoords="offset points", xytext=(4, 0), va="center", fontsize=9, color=GRIS_TEXT)
estilo(ax, "Valor unitario de las rosas colombianas por destino, 2025", "USD por tonelada",
       "Trade Map (ITC), 2025.", eje_x="USD/t", rejilla="x")
guardar(fig, "g7_valor_unitario_destinos")
exportar(vu_destinos, "g7_valor_unitario_destinos")

**Análisis del equipo:**

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 7. Cuota en la oferta mundial

In [ ]:
oferta = exportadores.nlargest(10, "valor_kusd")[["pais", "valor_kusd", "participacion_mundial_pct", "crec_valor_5a_pct"]]
hhi_oferta = hhi(exportadores["valor_kusd"])
print(f"HHI de la oferta mundial de rosas, 2025: {hhi_oferta:.4f}  (número equivalente de exportadores: {numeros_equivalentes(hhi_oferta):.1f})")
fig, ax = plt.subplots(figsize=(9, 5))
orden = oferta.sort_values("participacion_mundial_pct")
colores = [NARANJA if p == "Colombia" else GRIS_MID for p in orden["pais"]]
ax.barh(orden["pais"], orden["participacion_mundial_pct"], color=colores, height=0.6)
for y, v in enumerate(orden["participacion_mundial_pct"]):
    ax.annotate(f"{v:.1f} %", (v, y), textcoords="offset points", xytext=(4, 0), va="center", fontsize=9, color=GRIS_TEXT)
estilo(ax, "Diez mayores exportadores de rosas del mundo, 2025", "Participación en el valor exportado mundial",
       "Trade Map (ITC), 2025.", eje_x="% del valor mundial", rejilla="x")
guardar(fig, "g7_oferta_mundial")
exportar(oferta, "g7_oferta_mundial")

**Análisis del equipo:**

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 8. Ventaja comparativa revelada: RCA de Balassa, RSCA y NRCA

El numerador sale de las descargas del equipo; el denominador mundial (exportaciones totales de todos los productos) sale de la canasta del curso `mundo_exp_productos_hs2_serie.csv`, fila TOTAL.

In [ ]:
X_ij = mundo_2025["valor_kusd"]                                   # Colombia exporta rosas, 2025
X_i  = separar_mundo(exp_all)[0]["valor_kusd"]                    # Colombia exporta todo, 2025
X_j  = mundo_oferta["valor_kusd"]                                  # el mundo exporta rosas, 2025
X_w  = fila_canasta(canasta_mundo_exp, "TOTAL")["2025"]            # el mundo exporta todo, 2025

vcr = pd.DataFrame({
    "indice": ["RCA (Balassa)", "RSCA (Laursen)", "NRCA (Yu, Cai y Leung) x 10^4"],
    "valor": [float(rca_balassa(X_ij, X_i, X_j, X_w)),
              float(rsca_laursen(rca_balassa(X_ij, X_i, X_j, X_w))),
              float(nrca(X_ij, X_i, X_j, X_w)) * 1e4],
    "umbral_neutral": [1, 0, 0],
})
print(f"X_ij = {X_ij:,.0f}   X_i = {X_i:,.0f}   X_j = {X_j:,.0f}   X_w = {X_w:,.0f}   (USD miles)")
exportar(vcr, "g7_vcr_2025")
vcr

In [ ]:
# La misma familia para el capitulo 06 completo, 2021-2025, con la canasta del curso
vcr_serie = pd.DataFrame({"anio": [int(a) for a in ANIOS_CANASTA]})
vcr_serie["RCA_cap06"] = rca_balassa(fila_canasta(canasta_co_exp, "06"), fila_canasta(canasta_co_exp, "TOTAL"),
                                     fila_canasta(canasta_mundo_exp, "06"), fila_canasta(canasta_mundo_exp, "TOTAL"))
vcr_serie["RSCA_cap06"] = rsca_laursen(vcr_serie["RCA_cap06"])
exportar(vcr_serie, "g7_vcr_serie_cap06")

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(vcr_serie["anio"], vcr_serie["RSCA_cap06"], color=VERDE, linewidth=2.2, marker="o", markersize=6, label="Capítulo 06, RSCA")
ax.scatter([2025], [vcr.loc[1, "valor"]], color=AZUL, s=70, zorder=3, label="Rosas HS 060311, RSCA 2025")
for _, fila in vcr_serie.iterrows():
    ax.annotate(f"{fila['RSCA_cap06']:.2f}", (fila["anio"], fila["RSCA_cap06"]), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=9, color=GRIS_TEXT)
ax.set_ylim(-1.05, 1.05)
ax.axhline(0, color=GRIS_EJE, linewidth=0.8)
ax.set_xticks(vcr_serie["anio"])
ax.legend(frameon=False, fontsize=9, loc="lower left")
estilo(ax, "Ventaja comparativa revelada simétrica (RSCA)", "0 = neutral; +1 = especialización máxima",
       "Trade Map (ITC), 2025; canasta HS2 del curso.", eje_y="RSCA")
guardar(fig, "g7_rsca")
vcr_serie

**Análisis del equipo:** ¿La ventaja es de las rosas en particular o del sector floricultor completo?

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 9. Índice de Lafay

In [ ]:
# (a) Las rosas dentro del comercio total de Colombia, 2025
M_ij = mundo_imp_2025["valor_kusd"]
M_i  = separar_mundo(imp_all)[0]["valor_kusd"]
ibcr_rosas = (X_ij - M_ij) / (X_ij + M_ij)
ibcr_pais  = (X_i - M_i) / (X_i + M_i)
peso_rosas = (X_ij + M_ij) / (X_i + M_i)
lfi_rosas  = 100 * (ibcr_rosas - ibcr_pais) * peso_rosas
print(f"Lafay rosas 2025: IBCR producto = {ibcr_rosas:+.4f} | IBCR pais = {ibcr_pais:+.4f} | peso = {peso_rosas:.4%} | LFI = {lfi_rosas:+.4f}")

# (b) Lafay de los 97 capitulos de la canasta, 2025, para ubicar el capitulo 06 en contexto
capitulos = canasta_co_exp[canasta_co_exp["codigo"] != "TOTAL"][["codigo", "producto", "2025"]].rename(columns={"2025": "X"})
capitulos = capitulos.merge(canasta_co_imp[["codigo", "2025"]].rename(columns={"2025": "M"}), on="codigo", how="inner")
capitulos["LFI"] = lafay(capitulos["X"], capitulos["M"])
capitulos["producto"] = capitulos["producto"].str.slice(0, 45)
ranking_lafay = capitulos.sort_values("LFI", ascending=False).reset_index(drop=True)
exportar(ranking_lafay, "g7_lafay_capitulos")
ranking_lafay.head(10)

In [ ]:
extremos = pd.concat([ranking_lafay.head(8), ranking_lafay.tail(5)]).sort_values("LFI")
fig, ax = plt.subplots(figsize=(9, 6))
colores = [NARANJA if c == "06" else (AZUL if v >= 0 else ROJO) for c, v in zip(extremos["codigo"], extremos["LFI"])]
ax.barh(extremos["codigo"] + "  " + extremos["producto"], extremos["LFI"], color=colores, height=0.6)
ax.axvline(0, color=GRIS_EJE, linewidth=0.8)
estilo(ax, "Índice de Lafay por capítulo HS2, Colombia 2025", "Ocho mayores y cinco menores; el capítulo 06 resaltado",
       "Trade Map (ITC), canasta HS2 del curso.", eje_x="LFI", rejilla="x")
guardar(fig, "g7_lafay_capitulos")

**Análisis del equipo:**

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 10. Matriz de decisión 0-100 y prueba de sensibilidad

La ficha define cuatro dimensiones con pesos 35 / 25 / 20 / 20 y un semáforo **A Crecer / B Defender / C Pilotear / D Monitorear**. Los puntajes de cada mercado (0 a 100) son **juicio del equipo**, apoyado en las tablas anteriores. La tabla siguiente reúne los datos disponibles para cada candidato; después viene la celda editable.

In [ ]:
CANDIDATOS = ["Estados Unidos de América", "Canadá", "Reino Unido", "Países Bajos", "Japón"]
datos_candidatos = mercados[mercados["pais"].isin(CANDIDATOS)].set_index("pais").reindex(CANDIDATOS)
datos_candidatos = datos_candidatos.join(tabla_cagr.set_index("pais")[["cagr_2016_2025_pct"]])
exportar(datos_candidatos.reset_index(), "g7_datos_candidatos")
datos_candidatos

In [ ]:
# ====================== EDITABLE: pesos y puntajes 0-100 ======================
PESOS = {                                     # los de la ficha; se normalizan solos
    "Competitividad revelada": 35,
    "Atractivo de mercado":    25,
    "Economía de la operación": 20,
    "Acceso y riesgo":          20,
}
PUNTAJES = pd.DataFrame({                     # filas = dimensiones, columnas = mercados; 0 a 100
    "Estados Unidos de América": [50, 50, 50, 50],
    "Canadá":                    [50, 50, 50, 50],
    "Reino Unido":               [50, 50, 50, 50],
    "Países Bajos":              [50, 50, 50, 50],
    "Japón":                     [50, 50, 50, 50],
}, index=list(PESOS.keys()))
UMBRALES = {"A Crecer": 75, "B Defender": 60, "C Pilotear": 45}   # por debajo del ultimo: D Monitorear
# ==============================================================================

def semaforo(total):
    for etiqueta, umbral in UMBRALES.items():
        if total >= umbral:
            return etiqueta
    return "D Monitorear"

matriz = matriz_multicriterio(PUNTAJES, PESOS)
matriz["Semáforo"] = matriz["Total"].apply(semaforo)
exportar(matriz.reset_index().rename(columns={"index": "mercado"}), "g7_matriz_decision")
matriz

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
acumulado = np.zeros(len(matriz))
for color, dimension in zip(CATEGORIAS, PESOS.keys()):
    ax.barh(matriz.index, matriz[dimension], left=acumulado, color=color, height=0.6, label=dimension, edgecolor="white", linewidth=1.5)
    acumulado += matriz[dimension].values
for y, (t, s) in enumerate(zip(matriz["Total"], matriz["Semáforo"])):
    ax.annotate(f"{t:.0f}  {s}", (t, y), textcoords="offset points", xytext=(5, 0), va="center", fontsize=9, color=GRIS_TEXT)
ax.invert_yaxis()
ax.set_xlim(0, 115)
leyenda_fuera(ax)
estilo(ax, "Puntaje ponderado por mercado (0-100)", "Contribución de cada dimensión según los pesos de la ficha",
       "Elaboración del equipo.", eje_x="Puntaje", rejilla="x")
guardar(fig, "g7_matriz_decision")

In [ ]:
# Prueba de sensibilidad: se mueve cada peso +/- 5 puntos (redistribuyendo el resto) y se mira si cambia el orden
def sensibilidad(puntajes, pesos, delta=5):
    base = matriz_multicriterio(puntajes, pesos).index.tolist()
    filas = []
    for dimension in pesos:
        for signo in (+delta, -delta):
            alterados = dict(pesos)
            alterados[dimension] = max(0, alterados[dimension] + signo)
            orden = matriz_multicriterio(puntajes, alterados).index.tolist()
            filas.append({"peso_modificado": dimension, "cambio": f"{signo:+d}",
                          "primero": orden[0], "orden_completo": " > ".join(orden),
                          "cambia_el_lider": orden[0] != base[0], "cambia_el_orden": orden != base})
    return pd.DataFrame(filas)

prueba = sensibilidad(PUNTAJES, PESOS)
print("Orden base:", " > ".join(matriz.index))
print(f"El líder cambia en {prueba['cambia_el_lider'].sum()} de {len(prueba)} escenarios; el orden completo cambia en {prueba['cambia_el_orden'].sum()}.")
exportar(prueba, "g7_sensibilidad")
prueba

**Análisis del equipo:** Según la regla de la ficha: si una pequeña modificación cambia el ranking, la conclusión es de confianza media. ¿Qué confianza tiene la suya?

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# Entrega

1. Revisa que todas las celdas **Análisis del equipo** tengan texto.
2. `Archivo → Descargar → Descargar .ipynb`.
3. Sube el archivo al aula virtual. Las figuras y tablas de `salidas/` pueden ir al informe escrito.

# Referencias

Balassa, B. (1965). Trade liberalisation and "revealed" comparative advantage. *The Manchester School, 33*(2), 99–123.

Yu, R., Cai, J., & Leung, P. (2009). The normalized revealed comparative advantage index. *The Annals of Regional Science, 43*(1), 267–282.

Lafay, G. (1992). The measurement of revealed comparative advantages. En M. G. Dagenais & P. A. Muet (Eds.), *International Trade Modelling* (pp. 209–234). Chapman & Hall.

Baena-Rojas, J. J., & Cano, J. A. (2026). International market selection for exports of goods: A data analysis technique for organizational decision-making. *Global Business Review*. https://doi.org/10.1177/09721509261464305

International Trade Centre. (2025). *Trade Map: Trade statistics for international business development*. https://www.trademap.org

Durán Lima, J. E. (s.f.). *Indicadores de comercio exterior y política comercial: Generalidades metodológicas e indicadores básicos*. CEPAL.

McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a

Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55